# 09 · Performance Optimization, NumPy & Calling C (`ctypes`, `cffi`, mypyc, Cython)

**Module 10 – Advanced Python: Internals & Metaprogramming** · Estimated time: 60 min

### 🎯 What you will learn
- The optimisation workflow: **measure → find the hotspot → fix the algorithm → micro-optimise → go native**
- Python-level speedups with measurements: data structures, built-ins, comprehensions, string joins, caching, local names
- **Vectorisation with NumPy**
- Calling C libraries with **`ctypes`** and **`cffi`**
- Compiling Python with **mypyc**; overview of Cython, Numba, Nuitka, PyPy, Rust/PyO3
- When to use concurrency instead (Module 13)

In [1]:
import timeit, sys, platform
def bench(label, stmt, number=200, **ns):
    t = min(timeit.repeat(stmt, globals={**globals(), **ns}, number=number, repeat=3))
    print(f"{label:<38} {t / number * 1e6:>10.1f} µs")
    return t
print(platform.python_implementation(), sys.version.split()[0])

CPython 3.15.0


## 1. The workflow

1. **Is it actually slow?** Define a target ("this report must take < 2 s").
2. **Profile** (Module 12 · 05: `cProfile`/`profiling.tracing`, Tachyon sampling profiler) to find the 5% of code taking 95% of the time.
3. Fix the **algorithm / data structure** first: O(n²) → O(n log n) beats any micro-optimisation.
4. Use **built-ins and libraries** written in C (`sum`, `sorted`, `set`, `collections`, NumPy, pandas, polars).
5. Only then micro-optimise, parallelise (Module 13), or move hot code to compiled code.

## 2. Algorithms & data structures dominate

In [2]:
items = list(range(20_000))
lookup = set(items)
targets = list(range(0, 40_000, 7))

bench("membership in list (O(n) each)", "sum(t in items for t in targets)", number=3)
bench("membership in set  (O(1) each)", "sum(t in lookup for t in targets)", number=3)

membership in list (O(n) each)           396533.9 µs
membership in set  (O(1) each)              110.6 µs


0.0003318329981993884

In [3]:
words = [f"w{i % 500}" for i in range(50_000)]

def count_slow(ws):                      # O(n * k)
    return {w: ws.count(w) for w in set(ws)}

from collections import Counter
bench("count with list.count per word", "count_slow(words)", number=3)
bench("count with collections.Counter", "Counter(words)", number=3)

count with list.count per word           144792.1 µs
count with collections.Counter             1149.6 µs


0.0034489170066080987

## 3. Built-ins, comprehensions & idioms

In [4]:
nums = list(range(100_000))

def loop_sum(xs):
    total = 0
    for x in xs:
        total += x
    return total

bench("manual for-loop sum", "loop_sum(nums)", number=50)
bench("built-in sum()", "sum(nums)", number=50)

def squares_loop(xs):
    out = []
    for x in xs:
        out.append(x * x)
    return out

bench("append in a loop", "squares_loop(nums)", number=50)
bench("list comprehension", "[x * x for x in nums]", number=50)

parts = [str(i) for i in range(10_000)]
def concat(ps):
    s = ""
    for p in ps:
        s += p
    return s
bench("str += in a loop", "concat(parts)", number=50)
bench("''.join()", "''.join(parts)", number=50)

manual for-loop sum                        1326.7 µs
built-in sum()                              229.3 µs


append in a loop                           2011.2 µs


list comprehension                         1691.0 µs
str += in a loop                            276.7 µs
''.join()                                    37.4 µs


0.0018692919984459877

| Do | Instead of |
|---|---|
| `sum`, `min`, `max`, `any`, `all`, `sorted`, `map` | hand-written loops |
| comprehensions / generator expressions | `append` in loops |
| `"".join(parts)` | `+=` on strings |
| `set` / `dict` lookups | `x in list` repeatedly |
| `collections.Counter`, `defaultdict`, `deque` | manual bookkeeping |
| `@functools.cache` for repeated pure calls | recomputation |
| `__slots__` for millions of small objects | dict-backed instances |
| generators / streaming | loading everything into memory |
| local variables inside hot loops | repeated global/attribute lookups (smaller win since 3.11) |

## 4. Vectorisation with NumPy

NumPy stores numbers in compact C arrays and runs operations in **compiled loops**:

In [5]:
import numpy as np

py_list = list(range(1_000_000))
arr = np.arange(1_000_000)

bench("Python: [x*2+1 for x in list]", "[x * 2 + 1 for x in py_list]", number=5)
bench("NumPy:  arr*2+1 (vectorised)", "arr * 2 + 1", number=5)

data = np.random.default_rng(0).normal(50, 10, 1_000_000)
print("mean, std, 95th percentile:", data.mean().round(2), data.std().round(2), np.percentile(data, 95).round(2))
print("values > 70:", (data > 70).sum())          # boolean masks instead of loops

Python: [x*2+1 for x in list]             21737.9 µs
NumPy:  arr*2+1 (vectorised)                414.2 µs
mean, std, 95th percentile: 50.01 10.01 66.47
values > 70: 23017


For tabular data use **pandas** or **polars** (built on the same idea). Don't loop over NumPy arrays element by element; that's slower than plain lists!

## 5. Calling C directly with `ctypes` (standard library)

`ctypes` loads shared libraries (`.so`/`.dylib`/`.dll`) and calls their functions; no compiling needed.

In [6]:
import ctypes, ctypes.util

libm = ctypes.CDLL(ctypes.util.find_library("m"))       # the C math library
libm.cos.argtypes = [ctypes.c_double]                   # ALWAYS declare argument/return types
libm.cos.restype = ctypes.c_double
print("C cos(0) =", libm.cos(0.0))

libc = ctypes.CDLL(ctypes.util.find_library("c"))
libc.strlen.argtypes = [ctypes.c_char_p]
libc.strlen.restype = ctypes.c_size_t
print("C strlen('hello') =", libc.strlen(b"hello"))

C cos(0) = 1.0
C strlen('hello') = 5


C structs, arrays and pointers:

In [7]:
class Point(ctypes.Structure):
    _fields_ = [("x", ctypes.c_double), ("y", ctypes.c_double)]

p = Point(3.0, 4.0)
print(p.x, p.y, ctypes.sizeof(p), "bytes")

IntArray5 = ctypes.c_int * 5
arr5 = IntArray5(5, 1, 4, 2, 3)
CMPFUNC = ctypes.CFUNCTYPE(ctypes.c_int, ctypes.POINTER(ctypes.c_int), ctypes.POINTER(ctypes.c_int))
def py_cmp(a, b):                         # a Python callback called FROM C
    return a[0] - b[0]
libc.qsort.argtypes = [ctypes.c_void_p, ctypes.c_size_t, ctypes.c_size_t, CMPFUNC]
libc.qsort(arr5, len(arr5), ctypes.sizeof(ctypes.c_int), CMPFUNC(py_cmp))
print("sorted by C's qsort:", list(arr5))

3.0 4.0 16 bytes
sorted by C's qsort: [1, 2, 3, 4, 5]


> ⚠️ Wrong `argtypes`/`restype` can crash the interpreter (segfault). ctypes is powerful but unsafe.

## 6. `cffi`: a friendlier C interface

`pip install cffi`. You paste C declarations; cffi handles the conversions:

In [8]:
from cffi import FFI

ffi = FFI()
ffi.cdef("""
    double sqrt(double x);
    int abs(int x);
    size_t strlen(const char *s);
""")
C = ffi.dlopen(None)                      # the standard C library already loaded in the process
print(C.sqrt(2.0), C.abs(-42), C.strlen(b"python"))

1.4142135623730951 42 6


cffi also has an **API mode** that compiles a real extension module (faster, safer); it's used by `cryptography` and others.

## 7. Compiling Python with mypyc

**mypyc** (shipped with mypy) compiles **type-annotated** Python modules into C extensions, often 2–5× faster for typed, loop-heavy code (mypy itself is compiled with it).

In [9]:
import pathlib, subprocess, importlib, shutil, os
pathlib.Path("fastfib.py").write_text('''
def fib(n: int) -> int:
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

def count_primes(limit: int) -> int:
    count = 0
    for n in range(2, limit):
        d = 2
        is_prime = True
        while d * d <= n:
            if n % d == 0:
                is_prime = False
                break
            d += 1
        if is_prime:
            count += 1
    return count
''')
import fastfib as pure
bench("pure Python count_primes(30_000)", "pure.count_primes(30_000)", number=3, pure=pure)

result = subprocess.run([sys.executable, "-m", "mypyc", "fastfib.py"], capture_output=True, text=True)
built = [f for f in os.listdir(".") if f.startswith("fastfib") and f.endswith((".so", ".pyd"))]
print("compiled extension:", built or result.stderr[-500:])

pure Python count_primes(30_000)          12003.5 µs

compiled extension: ['fastfib.cpython-315-darwin.so']


In [10]:
if built:
    code = '''
import sys, timeit, os
sys.path.insert(0, os.getcwd())
import fastfib
print("loaded from:", os.path.basename(fastfib.__file__))
t = min(timeit.repeat("fastfib.count_primes(30_000)", globals=globals(), number=3, repeat=3)) / 3
print(f"mypyc-compiled count_primes(30_000)    {t * 1e6:>10.1f} µs")
'''
    print(subprocess.run([sys.executable, "-c", code], capture_output=True, text=True).stdout)

loaded from: fastfib.cpython-315-darwin.so
mypyc-compiled count_primes(30_000)        2139.2 µs



(We ran the compiled version in a fresh process because this notebook had already imported the pure-Python module.)

## 8. The native-speed toolbox

| Tool | What it is | Typical use |
|---|---|---|
| **NumPy / pandas / polars** | vectorised C/Rust libraries | numeric & tabular data |
| **mypyc** | compiles typed Python to C | speed up typed pure-Python modules |
| **Cython** | Python superset with C types → C extension | hot loops, wrapping C libraries |
| **Numba** | JIT for numeric Python functions (`@njit`) | NumPy-heavy loops, GPU |
| **PyO3 + maturin** | write extensions in **Rust** | safe, fast extensions (Ruff, Pydantic-core, polars use Rust) |
| **pybind11 / nanobind** | C++ bindings | wrapping C++ libraries |
| **ctypes / cffi** | call existing C libraries | no compilation (ctypes) |
| **PyPy** | alternative interpreter with a tracing JIT | long-running pure-Python apps |
| **Nuitka** | compiles a whole program to an executable | distribution + speed |
| **CPython JIT** (3.13+, improved in 3.15) | built-in experimental JIT | free speed-ups over time |

**Cython** example (`fib.pyx`, built with `cythonize`):
```cython
def fib(int n):
    cdef long a = 0, b = 1
    for _ in range(n):
        a, b = b, a + b
    return a
```

**Numba** example:
```python
from numba import njit
@njit
def total(arr):
    s = 0.0
    for x in arr:
        s += x
    return s
```

## 9. Use more cores
CPU-bound work can use multiple cores with `ProcessPoolExecutor`, `InterpreterPoolExecutor` (3.14) or a free-threaded build; see Module 13.

## 📝 Summary
- Measure first; optimise the hotspot; better algorithms beat micro-tweaks.
- Prefer built-ins, comprehensions, `join`, sets/dicts, `collections`, caching.
- NumPy vectorisation gives 10–100× for numeric work.
- `ctypes`/`cffi` call C libraries; mypyc/Cython/Numba/Rust compile hot code.

## 🏋️ Practice
1. Benchmark three ways to remove duplicates while keeping order: a loop with a list, a loop with a set, and `dict.fromkeys`.
2. Rewrite a nested-loop "find pairs that sum to target" from O(n²) to O(n) with a set, and measure both.
3. Use `ctypes` to call `libc.abs` and `libm.pow`.
4. Compile one of your typed modules with `python -m mypyc yourmodule.py` and compare timings.

<details><summary>Show solution for 2</summary>

```python
def pairs_slow(nums, target):
    return [(a, b) for i, a in enumerate(nums) for b in nums[i + 1:] if a + b == target]

def pairs_fast(nums, target):
    seen, out = set(), []
    for n in nums:
        if target - n in seen:
            out.append((target - n, n))
        seen.add(n)
    return out

import random, timeit
nums = random.sample(range(100_000), 3_000)
print(timeit.timeit(lambda: pairs_slow(nums, 50_000), number=1))
print(timeit.timeit(lambda: pairs_fast(nums, 50_000), number=1))
```
</details>